In [47]:
import os
import pandas as pd
import re

print("CleanSheet libraries imported successfully!")

CleanSheet libraries imported successfully!


In [48]:
from pathlib import Path

file_path = Path("../data/raw/iris_dataset.csv")

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

display(df.head())

Dataset loaded successfully!
Number of rows: 150
Number of columns: 5


,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [49]:
def audit_csv(df):
    report = {}

    report["rows"] = len(df)
    report["columns"] = len(df.columns)
    report["duplicate_rows"] = int(df.duplicated().sum())

    report["empty_rows"] = int(df.isna().all(axis=1).sum())
    report["empty_columns"] = [
        col for col in df.columns
        if df[col].isna().all()
    ]

    report["missing_values"] = df.isna().sum().to_dict()
    report["column_types"] = df.dtypes.astype(str).to_dict()

    # Check invalid email addresses
    email_columns = [
        col for col in df.columns
        if col.lower() == "email"
    ]

    report["invalid_emails"] = {}

    for col in email_columns:
        email_values = df[col].dropna().astype(str)

        invalid_count = sum(
            not re.fullmatch(
                r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}",
                email
            )
            for email in email_values
        )

        report["invalid_emails"][col] = invalid_count

    # Check negative ages
    age_columns = [
        col for col in df.columns
        if col.lower() == "age"
    ]

    report["invalid_ages"] = {}

    for col in age_columns:
        ages = pd.to_numeric(df[col], errors="coerce")
        report["invalid_ages"][col] = int((ages < 0).sum())

    return report

In [50]:
def display_report(report):
    print("\n========== CLEANSheet AUDIT REPORT ==========")

    print("Total rows:", report["rows"])
    print("Total columns:", report["columns"])
    print("Duplicate rows:", report["duplicate_rows"])
    print("Completely empty rows:", report["empty_rows"])
    print("Completely empty columns:", report["empty_columns"])

    print("\nMissing values by column:")
    for col, count in report["missing_values"].items():
        print(f"{col}: {count}")

    print("\nColumn data types:")
    for col, dtype in report["column_types"].items():
        print(f"{col}: {dtype}")

    print("\nInvalid emails:")
    if report["invalid_emails"]:
        for col, count in report["invalid_emails"].items():
            print(f"{col}: {count}")
    else:
        print("No email column found.")

    print("\nInvalid ages:")
    if report["invalid_ages"]:
        for col, count in report["invalid_ages"].items():
            print(f"{col}: {count}")
    else:
        print("No age column found.")

    print("=============================================")

In [51]:
original_report = audit_csv(df)
display_report(original_report)


========== CLEANSheet AUDIT REPORT ==========
Total rows: 150
Total columns: 5
Duplicate rows: 3
Completely empty rows: 0
Completely empty columns: []

Missing values by column:
sepal_length: 0
sepal_width: 0
petal_length: 0
petal_width: 0
species: 0

Column data types:
sepal_length: float64
sepal_width: float64
petal_length: float64
petal_width: float64
species: object

Invalid emails:
No email column found.

Invalid ages:
No age column found.


In [52]:
test_df = pd.DataFrame({
    "name": ["Anu", "Ravi", "Anu", "Priya", "Kiran"],
    "age": [20, -5, 20, None, 22],
    "email": [
        "anu@gmail.com",
        "invalid-email",
        "anu@gmail.com",
        "priya@gmail.com",
        None
    ]
})

print("Original test dataset:")
display(test_df)

print("\nTest dataset audit:")
display_report(audit_csv(test_df))

Original test dataset:


,name,age,email
0,Anu,20.0,anu@gmail.com
1,Ravi,-5.0,invalid-email
2,Anu,20.0,anu@gmail.com
3,Priya,NaN,priya@gmail.com
4,Kiran,22.0,None



Test dataset audit:

========== CLEANSheet AUDIT REPORT ==========
Total rows: 5
Total columns: 3
Duplicate rows: 1
Completely empty rows: 0
Completely empty columns: []

Missing values by column:
name: 0
age: 1
email: 1

Column data types:
name: object
age: float64
email: object

Invalid emails:
email: 1

Invalid ages:
age: 1


In [53]:
def clean_csv(df):
    cleaned_df = df.copy()

    # 1. Remove duplicate rows
    cleaned_df = cleaned_df.drop_duplicates()

    # 2. Remove completely empty rows
    cleaned_df = cleaned_df.dropna(how="all")

    # 3. Remove completely empty columns
    cleaned_df = cleaned_df.dropna(axis=1, how="all")

    # 4. Clean the age column, if present
    age_columns = [
        col for col in cleaned_df.columns
        if col.lower() == "age"
    ]

    for col in age_columns:
        cleaned_df[col] = pd.to_numeric(
            cleaned_df[col], errors="coerce"
        )

        # Replace negative ages with missing values
        cleaned_df.loc[cleaned_df[col] < 0, col] = pd.NA

        # Fill missing ages with the median
        if cleaned_df[col].notna().any():
            median_age = cleaned_df[col].median()
            cleaned_df[col] = cleaned_df[col].fillna(median_age)

    # 5. Clean the email column, if present
    email_columns = [
        col for col in cleaned_df.columns
        if col.lower() == "email"
    ]

    for col in email_columns:
        email_pattern = (
            r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
        )

        valid_email = cleaned_df[col].apply(
            lambda x: (
                isinstance(x, str)
                and re.fullmatch(email_pattern, x) is not None
            )
        )

        cleaned_df.loc[~valid_email, col] = pd.NA

    return cleaned_df

In [54]:
test_clean = clean_csv(test_df)

print("Before cleaning:", test_df.shape)
print("After cleaning:", test_clean.shape)

print("\nCleaned test dataset:")
display(test_clean)

print("\nAudit report after cleaning:")
display_report(audit_csv(test_clean))

Before cleaning: (5, 3)
After cleaning: (4, 3)

Cleaned test dataset:


,name,age,email
0,Anu,20.0,anu@gmail.com
1,Ravi,21.0,<NA>
3,Priya,21.0,priya@gmail.com
4,Kiran,22.0,<NA>



Audit report after cleaning:

========== CLEANSheet AUDIT REPORT ==========
Total rows: 4
Total columns: 3
Duplicate rows: 0
Completely empty rows: 0
Completely empty columns: []

Missing values by column:
name: 0
age: 0
email: 2

Column data types:
name: object
age: float64
email: object

Invalid emails:
email: 0

Invalid ages:
age: 0


In [55]:
iris_clean = clean_csv(df)

print("Original dataset shape:", df.shape)
print("Cleaned dataset shape:", iris_clean.shape)

print("\nFirst five rows of the cleaned dataset:")
display(iris_clean.head())

print("\nAudit report after cleaning:")
display_report(audit_csv(iris_clean))
output_path = Path("../data/processed/iris_cleaned.csv")

iris_clean.to_csv(output_path, index=False)

print("File saved successfully!")
print("File exists:", os.path.exists(output_path))

Original dataset shape: (150, 5)
Cleaned dataset shape: (147, 5)

First five rows of the cleaned dataset:


,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa



Audit report after cleaning:

========== CLEANSheet AUDIT REPORT ==========
Total rows: 147
Total columns: 5
Duplicate rows: 0
Completely empty rows: 0
Completely empty columns: []

Missing values by column:
sepal_length: 0
sepal_width: 0
petal_length: 0
petal_width: 0
species: 0

Column data types:
sepal_length: float64
sepal_width: float64
petal_length: float64
petal_width: float64
species: object

Invalid emails:
No email column found.

Invalid ages:
No age column found.
File saved successfully!
File exists: True


In [56]:
saved_df = pd.read_csv("../data/processed/iris_cleaned.csv")

print("===== FINAL CLEANSHEET VERIFICATION =====")
print("File loaded successfully:", True)
print("Dataset shape:", saved_df.shape)
print("Duplicate rows:", saved_df.duplicated().sum())
print("Total missing values:", saved_df.isna().sum().sum())

print("\nFinal audit report:")
display_report(audit_csv(saved_df))

===== FINAL CLEANSHEET VERIFICATION =====
File loaded successfully: True
Dataset shape: (147, 5)
Duplicate rows: 0
Total missing values: 0

Final audit report:

========== CLEANSheet AUDIT REPORT ==========
Total rows: 147
Total columns: 5
Duplicate rows: 0
Completely empty rows: 0
Completely empty columns: []

Missing values by column:
sepal_length: 0
sepal_width: 0
petal_length: 0
petal_width: 0
species: 0

Column data types:
sepal_length: float64
sepal_width: float64
petal_length: float64
petal_width: float64
species: object

Invalid emails:
No email column found.

Invalid ages:
No age column found.
